# LC 1143 — Longest Common Subsequence
**Difficulty:** Medium | **Pattern:** 2D DP on Strings

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> Build a 2D table where
<code>dp[i][j]</code> stores the LCS length for the first
<code>i</code> chars of <code>text1</code> and first <code>j</code>
chars of <code>text2</code>. Characters match → extend diagonal;
otherwise take the best of skipping one character from either string.
</div>

## Official Problem Statement

Given two strings `text1` and `text2`, return the length of their
**longest common subsequence**. If there is no common subsequence,
return `0`.

A **subsequence** of a string is a new string generated from the
original string with some characters (can be none) deleted without
changing the relative order of the remaining characters.

**Example:**
```
Input:  text1 = "abcde", text2 = "ace"
Output: 3   # "ace" is the LCS
```
**Constraints:** `1 <= text1.length, text2.length <= 1000`

## What This Is Actually Asking

Find the longest sequence of characters that appears in **both**
strings in the **same relative order**, but not necessarily
contiguous.

- `"abcde"` and `"ace"` share `a`, `c`, `e` → LCS length = **3**
- Unlike substrings, subsequences can skip characters
- We only need the **length**, not the actual sequence

Key observations:
- If last chars match: LCS = 1 + LCS of remaining prefixes
- If last chars differ: LCS = max(skip last of text1,
  skip last of text2)
- Subproblems overlap heavily → DP is ideal

## Walk Through an Example by Hand

`text1 = "abcde"`, `text2 = "ace"`

Start with a `(m+1) x (n+1)` table filled with zeros.

For each cell `(i, j)`:
- `i=1, j=1`: `text1[0]='a'` == `text2[0]='a'` → `dp[0][0]+1 = 1`
- `i=1, j=2`: `'a' != 'c'` → `max(dp[0][2], dp[1][1]) = max(0,1) = 1`
- `i=1, j=3`: `'a' != 'e'` → `max(dp[0][3], dp[1][2]) = 1`
- `i=3, j=2`: `'c' == 'c'` → `dp[2][1]+1 = 1+1 = 2`
- `i=5, j=3`: `'e' == 'e'` → `dp[4][2]+1 = 2+1 = 3` ✓

Answer at `dp[5][3] = 3`.

## The Picture

DP table for `text1="abcde"`, `text2="ace"`:

```
      ""  a   c   e
  ""  [ 0, 0,  0,  0 ]
  a   [ 0, 1,  1,  1 ]   <-- 'a'=='a' diagonal +1
  b   [ 0, 1,  1,  1 ]
  c   [ 0, 1,  2,  2 ]   <-- 'c'=='c' diagonal +1
  d   [ 0, 1,  2,  2 ]
  e   [ 0, 1,  2,  3 ]   <-- 'e'=='e' diagonal +1
               ^
               Answer = dp[5][3] = 3

  MATCH:  dp[i][j] = dp[i-1][j-1] + 1   (diagonal)
  NO MATCH: dp[i][j] = max(dp[i-1][j],  (skip text1 char)
                           dp[i][j-1])   (skip text2 char)
```

## When To Use This Pattern

Use **2D DP on strings** when:
- Problem asks about comparing/aligning two sequences
- Subproblems can be expressed as prefixes of both strings
- Decision at each step: match characters or skip one

**Classic signals:**
- "Longest common..." → LCS DP
- "Edit distance / minimum operations" → uses LCS or similar
- "Is string X a subsequence of Y" → can use DP or two pointers

**Related problems:** LC 583, LC 72 (Edit Distance),
LC 1092 (Shortest Common Supersequence)

## The Approach

**Bottom-up 2D DP:**

1. Create `dp` table of size `(m+1) x (n+1)`, initialized to `0`
   - Row 0 / Col 0 = base case (empty prefix → LCS = 0)
2. Fill left-to-right, top-to-bottom:
   - If `text1[i-1] == text2[j-1]`: `dp[i][j] = dp[i-1][j-1] + 1`
   - Else: `dp[i][j] = max(dp[i-1][j], dp[i][j-1])`
3. Return `dp[m][n]`

**Why this works:** Each cell `dp[i][j]` correctly stores the LCS
of `text1[:i]` and `text2[:j]` because all smaller subproblems
are solved before we need them (left, above, diagonal).

In [ ]:
from typing import List

In [ ]:
def test_harness(func):
    """
    Runs test cases for longestCommonSubsequence.
    Prints PASSED/FAILED for each, then a summary.
    """
    tests = [
        # (text1, text2, expected)
        ("abcde", "ace",   3),  # classic
        ("abc",   "abc",   3),  # identical strings
        ("abc",   "def",   0),  # no common chars
        ("a",     "a",     1),  # single matching char
        ("a",     "b",     0),  # single non-matching
        ("bl",    "yby",   1),  # partial overlap
        ("oxcpqrsvwf", "shmtulqrypy", 2),  # longer strings
    ]
    passed = 0
    for i, (t1, t2, exp) in enumerate(tests):
        got = func(t1, t2)
        status = "PASSED" if got == exp else "FAILED"
        if status == "PASSED":
            passed += 1
        print(
            f"[{status}] Test {i+1}: "
            f"text1={t1!r}, text2={t2!r} "
            f"| expected={exp}, got={got}"
        )
    total = len(tests)
    print(f"\nResult: {passed}/{total} tests passed.")

In [ ]:
def longestCommonSubsequence(text1: str, text2: str) -> int:
    """
    Returns the length of the longest common subsequence
    of text1 and text2.

    Approach: 2D bottom-up DP.
      dp[i][j] = LCS length of text1[:i] and text2[:j]
      If chars match:  dp[i][j] = dp[i-1][j-1] + 1
      Else:            dp[i][j] = max(dp[i-1][j], dp[i][j-1])

    Args:
        text1: First input string.
        text2: Second input string.

    Returns:
        Integer length of the LCS.

    Examples:
        >>> longestCommonSubsequence("abcde", "ace")
        3
        >>> longestCommonSubsequence("abc", "def")
        0
    """
    # Debug: print input sizes
    print(
        f"[DEBUG] text1 len={len(text1)}, "
        f"text2 len={len(text2)}"
    )
    pass

In [ ]:
# Uncomment and run when solution is ready
# test_harness(longestCommonSubsequence)

## Complexity

| | Value | Reason |
|---|---|---|
| **Time** | O(m × n) | Fill every cell of the 2D DP table |
| **Space** | O(m × n) | Store the full DP table |

**Space optimisation:** Only need the current and previous row
at any time → can reduce to O(min(m, n)) space using two 1D
arrays. The time complexity stays the same.

Where `m = len(text1)`, `n = len(text2)`.

## Real World Connection

**LCS is everywhere in software:**

- **`git diff`** — finding the LCS of two file versions to show
  minimal additions/deletions between commits
- **Bioinformatics** — aligning DNA/protein sequences to find
  evolutionary similarity (BLAST algorithm uses LCS variants)
- **Plagiarism detection** — measuring how similar two documents
  are by their longest shared subsequence
- **File synchronisation** — rsync and similar tools identify
  common blocks to minimise data transfer

The same 2D DP table is the backbone of the broader family
of sequence alignment algorithms.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra